# Fase 11 — lengan BiLSTM rujukan pada SEMUA lipatan (butuh GPU)

Tujuan: adu murni **resep kita vs metode paper** (SARIMA + BiLSTM) pada **lipatan dan origin yang identik**.

Protokol paper diambil apa adanya dari `notebooks/01_baseline_bilstm.ipynb`:
SARIMA `(2,1,1)(0,0,0,12)` → residu → MinMaxScaler → sekuens 90 langkah → BiLSTM 2 lapis → koreksi 1 langkah
→ `hybrid = SARIMA + koreksi`. Arsitektur tetapnya: 192/64 unit, dropout 0,4, AdamW, Huber, early-stopping.
Model ini punya **544.513 parameter** — sama persis dengan angka yang tercatat di `results/01_baseline_metrics.csv`,
jadi reproduksinya pas.

**Setelan runtime Kaggle:**
- Accelerator: **GPU T4** (atau T4 x2; kode pakai satu GPU)
- Internet: **On** (perlu clone repo publik)
- Perkiraan waktu: **±45 menit** untuk 15 model (3 polutan x 5 lipatan), 100 epoch maksimum tiap model
- Aman kalau sesi mati: bobot disimpan per (polutan, lipatan) dan hasil ditulis bertahap ke CSV;
  sekadar jalankan ulang sel untuk melanjutkan, bukan mulai dari nol


## Sel 1 — ambil kode + data

In [ ]:
import os, subprocess
REPO = 'https://github.com/madajabbar/hybrid-sarima-nbeats_Anc.git'
ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
DEST = os.path.join(ROOT, 'AnC')
if not os.path.isdir(DEST):
    subprocess.run(['git', 'clone', '-q', '-b', 'main', REPO, DEST], check=True)
else:
    subprocess.run(['git', '-C', DEST, 'pull', '-q'], check=False)
os.chdir(DEST)
print('cwd:', os.getcwd())
print('data:', os.path.isfile('data/transformed_data.csv'))
!nvidia-smi -L

## Sel 2 — (opsional) pasang keras-tuner

Hanya perlu kalau memakai `--tune once` atau `--tune each` (mengikuti prosedur tuning di paper).
Untuk jalur cepat (`--tune off`) sel ini boleh dilewati.

In [ ]:
%pip install -q keras-tuner
import keras_tuner; print('keras-tuner', keras_tuner.__version__)

## Sel 3 — jalankan lengan BiLSTM

Kenapa default `--tune off`: prosedur tuning paper (RandomSearch 20 percobaan per model) kalau
diulang untuk 15 model butuh belasan jam GPU. Supaya perbandingan tetap **terkendali dan seragam**,
arsitektur tetap hasil publikasi dipakai untuk semua lipatan. Ini dicatat di naskah.

Kalau ingin mengikuti protokol tuning: `--tune once` (tuning sekali per polutan, lalu dipakai ulang).

In [ ]:
!python scripts/13_bilstm_all_folds.py --tune off --epochs 100 2>&1 | grep -vE "oneDNN|cpu_feature|TF-TRT|WARNING" 

## Sel 4 — hitung adu finalnya (resep kita vs paper)

Menghasilkan `results/14_vs_bilstm_all_folds.csv`, `results/15_vs_bilstm_all_folds_dm.csv`,
dan tabel markdown `03_Catatan/HASIL_VS_BILSTM.md`.

In [ ]:
!python scripts/14_vs_bilstm_all_folds.py

## Sel 5 — cek kewajaran (sanity check)

Lipatan pertama NO2 sebelumnya sudah pernah dihitung di `results/01_baseline_horizon_metrics.csv`
(versi yang masih memakai tuning): RMSE 0,5915 (h1) · 0,7603 (h7) · 0,6289 (h30).
Angka baru boleh bergeser sedikit, tapi tidak boleh jauh — kalau jauh, ada yang salah setelan.

In [ ]:
import pandas as pd
old = pd.read_csv('results/01_baseline_horizon_metrics.csv')
old = old[(old.model == 'SARIMA+BiLSTM')][['horizon', 'rmse']].rename(columns={'rmse': 'versi_tuning_lama'})
new = pd.read_csv('results/13_bilstm_all_folds_predictions.csv')
new = new[(new.fold == 0) & (new.feature == 'NO2')]      # WAJIB: NO2/PM10/PM25 beda skala, jangan dicampur
rows = []
for h in (1, 7, 30):
    d = new[new[f'actual_h{h}'].notna()]
    rows.append({'horizon': h, 'RMSE_baru_fold0': float(((d[f'actual_h{h}'] - d[f'hybrid_h{h}'])**2).mean()**0.5)})
cmp = pd.DataFrame(rows).merge(old, on='horizon')
cmp['selisih_%'] = (cmp.RMSE_baru_fold0 / cmp.versi_tuning_lama - 1) * 100
print(cmp.round(4).to_string(index=False))
print()
print('selesai — jalankan sel 4 untuk tabel final, lalu unduh:')
print('  results/13_bilstm_all_folds_predictions.csv')
print('  results/14_vs_bilstm_all_folds.csv')
print('  results/15_vs_bilstm_all_folds_dm.csv')
print('  03_Catatan/HASIL_VS_BILSTM.md')